In [ ]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.interpolate import griddata
from scipy.linalg import lstsq
from scipy.ndimage import median_filter, vectorized_filter,binary_dilation

from skimage import transform
from skimage.measure import profile_line

import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

Traceback (most recent call last):
  File "c:\Users\bepoole\AppData\Local\miniforge3\envs\hrdic_process\Lib\site-packages\matplotlib\backends\backend_qt.py", line 523, in _draw_idle
    self.draw()
  File "c:\Users\bepoole\AppData\Local\miniforge3\envs\hrdic_process\Lib\site-packages\matplotlib\backends\backend_agg.py", line 382, in draw
    self.figure.draw(self.renderer)
  File "c:\Users\bepoole\AppData\Local\miniforge3\envs\hrdic_process\Lib\site-packages\matplotlib\artist.py", line 94, in draw_wrapper
    result = draw(artist, renderer, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bepoole\AppData\Local\miniforge3\envs\hrdic_process\Lib\site-packages\matplotlib\artist.py", line 71, in draw_wrapper
    return draw(artist, renderer)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bepoole\AppData\Local\miniforge3\envs\hrdic_process\Lib\site-packages\matplotlib\figure.py", line 3257, in draw
    mimage._draw_list_compositing_images(
  File "c

In [2]:
def lsm_read(lsm_file):
    # for reading in lsm output from ZEISS Confomap

    df = pd.read_csv(lsm_file,names=['x','y','z'])

    x = np.asarray(df['x'])
    y = np.asarray(df['y'])
    z = np.asarray(df['z'])

    # calculate shape - this must be done on the raw data 
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)


    # create new grid to interpolate data onto
    xg,yg = np.meshgrid(np.arange(x0,x1,x_step),np.arange(y0,y1,y_step))



    # remove the weird way that ConfoMaps saves non-measured points
    x = x[z !='***']
    y = y[z !='***']
    z = z[z !='***']


    # interpolate onto grid to produced gridded data
    zg = griddata(np.asarray([x,y]).T,z,(xg,yg),method='nearest')

    # flip up down for zg
    zg = np.flipud(zg)

    return xg, yg, zg, x_step

def resample_gridded_data(xg,yg,zg,new_step):

    # flatten arrays - we could probably use RegularGridInterpolator but this works for now
    x = xg.flatten()
    y = yg.flatten()
    z = zg.flatten()

    # calculate shape - this must be done on the raw data
    x0 = np.nanmin(x)
    x1 = np.nanmax(x)
    y0 = np.nanmin(y)
    y1 = np.nanmax(y)

    x_size = x1 - x0
    y_size = y1 - y0

    # calculate step size 
    x_step = np.round(np.min(np.abs(np.diff(x))),4)
    y_step = np.round(np.max(np.abs(np.diff(y))),4)

    # create new grid to interpolate data onto
    xg_new,yg_new = np.meshgrid(np.arange(x0,x1,new_step),np.arange(y0,y1,new_step))

    # interpolate onto grid to produced gridded data
    zg_new = griddata(np.asarray([x,y]).T,z,(xg_new,yg_new),method='nearest')

    return xg_new, yg_new, zg_new


In [3]:

# path to LSM file 
lsm_file = './LSM/raw_surface_v2.txt'

# import data
xg,yg,zg,lsm_step = lsm_read(lsm_file)




# # processing to get data into useful form 
# # crop the rubbish data from edges 
# xL = 800
# xR = 800
# yT = 800
# yB = 800

# # if we want to nanify the deleted data
# # zg[:,:xL] = np.nan
# # zg[:,-xR:] = np.nan

# # zg[:yT,:] = np.nan
# # zg[-yB:,:] = np.nan

# # plt.figure()
# # plt.imshow(zg)

# zg = zg[yT:-yB,xL:-xR]
# xg = xg[yT:-yB,xL:-xR]
# yg = yg[yT:-yB,xL:-xR]
# # plt.figure()
# # plt.imshow(zg)



# best-fit linear plane
A = np.c_[xg.flatten(),yg.flatten(), np.ones(xg.flatten().shape[0])]

C,_,_,_ = lstsq(A, zg.flatten())    # coefficients
    
# fitted plane
zg_fit = C[0]*xg + C[1]*yg + C[2]

# plt.figure()
fig,ax = plt.subplots(1,3)
ax[0].imshow(zg)
ax[0].set_title('Raw surface')
ax[1].imshow(zg_fit)
ax[1].set_title('Fitted flat plane')
ax[2].imshow(zg - zg_fit)
ax[2].set_title('Corrected')

plt.tight_layout()

# corrected surface
zg_flat = zg - zg_fit

# set lowest point on map to zero 
zg_flat = zg_flat - zg_flat.min()

In [4]:
# current dataset is overkill for DIC
# resample to DIC step size 

dic_step_px = 10
dic_px_size = 20/2048
dic_step = dic_step_px*dic_px_size # microns 



xg_new, yg_new, zg_new = resample_gridded_data(xg,yg,zg_flat,dic_step)



In [6]:
# median filter to smooth out bumps from speckle pattern

# # filter kernel size
# k_size = 15

# zg_filtered = median_filter(zg_new,k_size)
zg_filtered = gaussian_filter(zg_new,sigma=5)

fig,ax = plt.subplots()
surf = ax.imshow(zg_filtered,cmap='gist_grey',vmin=0,vmax=2)
bar = plt.colorbar(surf)
bar.set_label('Altitude / μm')

In [7]:
# zg_grad = np.gradient(zg_filtered)
zg_grad = np.gradient(zg_filtered)
zg_grad_mag = ((zg_grad[0]**2 + zg_grad[1]**2)**0.5)/dic_step
# zg_grad_mag = gaussian_filter(zg_grad_mag,sigma=2)

zg_grad_mag = median_filter(zg_grad_mag,10)

fig,ax = plt.subplots()
surf = ax.imshow(zg_grad_mag,cmap='gist_grey',vmin=0.02,vmax=0.2)
bar = plt.colorbar(surf)
bar.set_label('Altitude gradient magnitude / μm/μm')

In [8]:
exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

# for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
#     hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

dic_file = dic_step_list[-2]
hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)


In [9]:
# # clip points on DIC map - start bottom left and go ACW

# fig,ax = plt.subplots(1,2)
# surf = ax[0].imshow(zg_grad_mag,cmap='turbo',vmin=0,vmax=0.2)
# bar = plt.colorbar(surf)
# bar.set_label('Altitude gradient magnitude / μm/μm')


# ax[1].imshow(dic_map.data['max_shear'],vmin=0,vmax=0.1)

# dic_click = plt.ginput(5,show_clicks=True,timeout=-1)
# dic_click = np.asarray(dic_click)

# ax[1].plot(dic_click[:,0],dic_click[:,1],'rx')

In [10]:
# # # clip points on zmap map - start bottom left and go ACW

# fig,ax = plt.subplots(1,2)


# ax[1].imshow(dic_map.data['max_shear'],vmin=0,vmax=0.1)
# ax[1].plot(dic_click[:,0],dic_click[:,1],'rx')

# surf = ax[0].imshow(zg_grad_mag,cmap='gist_grey',vmin=0,vmax=0.1)
# bar = plt.colorbar(surf)
# bar.set_label('Altitude gradient magnitude / μm/μm')

# lsm_click = plt.ginput(5,show_clicks=True,timeout=-1)
# lsm_click = np.asarray(lsm_click)

# ax[0].plot(lsm_click[:,0],lsm_click[:,1],'rx')

In [11]:
lsm_click = ([[1927.00974462, 1721.95661122],
       [ 442.76008065, 2944.27986391],
       [2873.97446237, 2272.67368112],
       [3015.01176075,  586.9421623 ],
       [ 530.06888441,  969.75768649]])

dic_click = ([[1578.56451613, 1385.56214718],
       [ 178.35322581, 2692.72142137],
       [2513.51572581, 1957.16738911],
       [2619.8608871 ,  171.45488911],
       [ 196.07741935,  641.14601815]])

In [12]:
tf = transform.ProjectiveTransform()
tf.estimate(lsm_click,dic_click)

zg_grad_mag_warped = transform.warp(zg_grad_mag,tf.inverse,output_shape=dic_map.shape)
zg_filtered_warped = transform.warp(zg_filtered,tf.inverse,output_shape=dic_map.shape)

# we need to shift this to account for the crop in the DIC map
tf_shift = transform.AffineTransform(translation=(100,100))

zg_grad_mag_warped = transform.warp(zg_grad_mag_warped,tf_shift.inverse,output_shape=dic_map.shape)
zg_filtered_warped = transform.warp(zg_filtered_warped,tf_shift.inverse,output_shape=dic_map.shape)


C:\Users\bepoole\AppData\Local\Temp\ipykernel_13680\244900557.py:2: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `ProjectiveTransform.from_estimate` class constructor instead.
  tf.estimate(lsm_click,dic_click)


In [13]:
# surely it can't be this simple! 
# let's try to bodge it in to defdap 
dic_map.data.add(
    'lsm_height', zg_filtered_warped,
    unit='μm', type='map', order=0,
    plot_params={
        'plot_colour_bar': True,
        'clabel': 'Height',
    }
)

dic_map.data.add(
    'lsm_height_grad', zg_grad_mag_warped,
    unit='μm/μm', type='map', order=0,
    plot_params={
        'plot_colour_bar': True,
        'clabel': 'Height gradient',
    }
)


In [14]:

# link ebsd  map
ebsd_frame = experiment.Frame()
data_dir = Path('.')
ebsd.Map(data_dir / 'Pre_EBSD/map.cpr',
         increment=exp.increments[0], frame=ebsd_frame)

ebsd_map = exp.increments[0].maps['ebsd']
# ebsd_map.set_homog_point()

dic_map = exp.increments[0].maps['hrdic']

# dic_map.set_homog_point(vmin=0,vmax=0.05)

ebsd_frame.homog_points = [(1946, 1565),
 (2443, 1000),
 (1305, 1027),
 (1395, 2225),
 (2572, 2193),
 (1876, 1077),
 (2613, 1497),
 (1822, 2208),
 (1259, 1661),
 (2229, 1260),
 (1641, 1325),
 (1693, 1794),
 (2195, 1762)]

dic_frame.homog_points = [(1582, 1380),
 (2615, 172),
 (238, 226),
 (453, 2748),
 (2882, 2728),
 (1431, 326),
 (2970, 1240),
 (1329, 2732),
 (157, 1568),
 (2170, 731),
 (941, 863),
 (1061, 1854),
 (2100, 1795)]

ebsd_map = exp.increments[0].maps['ebsd']

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.link_ebsd_map(ebsd_map, transform_type="polynomial",order=2)
    # dic_map.link_ebsd_map(ebsd_map, transform_type="affine")

Loaded EBSD data (dimensions: 3727 x 2795 pixels, step size: 0.2 um)


In [18]:
fig,ax = plt.subplots(1,2)

dic_map.plot_map('max_shear', plot_gbs='line',plot_scale_bar=True,boundary_colour='white',cmap='viridis',vmin=0,vmax=0.1,fig=fig,ax=ax[0])
dic_map.plot_map('lsm_height_grad',plot_gbs='line',plot_scale_bar=True,boundary_colour='white',cmap='afmhot',vmin=0,vmax=0.1,fig=fig,ax=ax[1])

plt.tight_layout()

C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,


In [20]:
dic_map.plot_map('lsm_height',plot_gbs='line',plot_scale_bar=True,boundary_colour='white',cmap='afmhot',vmin=0,vmax=3)

C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,


C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,


Text(0.5, 0, 'Position along path / μm')

In [ ]:
def height_map_line_profiler(dic_map,click_on=True,line_profile_width=3,points=" "):
    # line profiles arrays of interest 
    ms_map = dic_map.data['max_shear']
    # ms_map = vectorized_filter(dic_map.data['max_shear'],function=np.nanmedian,size=3)
    ms_map[np.isnan(ms_map)] = 0.1
    # ms_map = median_filter(dic_map.data['max_shear'],31)

    # height maps
    height_map = dic_map.data['lsm_height']
    height_grad_map = dic_map.data['lsm_height_grad']

    # gb map
    gb_map = dic_map.data['grain_boundaries'].image

    # dilate gb_map 
    gb_map = binary_dilation(gb_map,iterations=2)
    # plt.figure()
    # plt.imshow(gb_map)


    # extract some line profiles 
    if click_on == True:

        plt.figure()
        # plt.imshow(dic_map.data['lsm_height2'],alpha=1,cmap='Blues')
        plt.imshow(ms_map,alpha=1,cmap='viridis',vmin=0,vmax=0.1)

        plt.xlim([0,2997])
        plt.ylim([2997,0])

        pts = plt.ginput(2,show_clicks=True,timeout=-1)

        pts = np.array(pts)

    else: 
        pts = points

    x0 = pts[0,0]
    x1 = pts[1,0]

    y0 = pts[0,1]
    y1 = pts[1,1]

    profile_length = dic_map.scale*((x1 - x0)**2 + (y1 - y0)**2)**0.5

    plt.plot((x0,x1),(y0,y1),'k-')
    plt.plot(x0,y0,'x')
    plt.plot(x1,y1,'o')
    plt.close(plt.gcf())


    #interpolate over datasets we want to get line profiles from 
    ms_profile = profile_line(ms_map,(y0,x0),(y1,x1),linewidth=line_profile_width)
    height_profile = profile_line(height_map,(y0,x0),(y1,x1),linewidth=line_profile_width)
    height_profile = height_profile - height_profile.min()

    height_grad_profile = profile_line(height_grad_map,(y0,x0),(y1,x1),linewidth=line_profile_width)

    gb_profile = profile_line(gb_map,(y0,x0),(y1,x1),linewidth=1)

    prof_pos = np.linspace(0,profile_length,len(ms_profile))

    gb_peaks = find_peaks(gb_profile)

        
    # do the plot 
    fig = plt.figure()

    fig.set_size_inches(15,5)

    gs = GridSpec(3,3,width_ratios=[4,5,5])
    ax0 = fig.add_subplot(gs[0,:1])
    ax1 = fig.add_subplot(gs[1,:1])
    ax2 = fig.add_subplot(gs[2,:1])
    ax3 = fig.add_subplot(gs[:,1:2])
    ax4 = fig.add_subplot(gs[:,2:])

    ax0.plot(prof_pos, ms_profile,'+-',markeredgecolor='teal',color='teal')
    ax0.set_ylabel(r'$ϵ_{\mathrm{eff}} / -$')


    ax1.plot(prof_pos, height_profile,'+-',markeredgecolor='orange',color='orange')
    ax1.set_ylabel(r'$z$ / μm')


    ax2.plot(prof_pos, height_grad_profile,'+-',markeredgecolor='darkorange',color='darkorange')
    ax2.set_ylabel(r'$|\nabla(z)|$ / μm/μm')


    # gb zone 
    gb_zone_width = 2 # microns 
    gbx0 = prof_pos[gb_peaks[0]] - gb_zone_width/2
    gbx1 = prof_pos[gb_peaks[0]] + gb_zone_width/2

    for ax in [ax0,ax1,ax2]:
        for i in range(len(gbx0)):
            ax.axvspan(gbx0[i],gbx1[i],alpha=0.2,color='purple')
        ax.grid()
        ax.set_ylim(bottom=0)

    ax2.set_xlabel('Position along path / μm')

    #change zero point of height map 

    dic_map.plot_map('max_shear', 
                plot_gbs='line',
                plot_scale_bar=True,
                boundary_colour='white',
                cmap='viridis',
                vmin=0,vmax=0.1,
                fig=fig,
                ax=ax3)

    dic_map.plot_map('lsm_height_grad', 
                    plot_gbs='line',
                    plot_scale_bar=True,
                    boundary_colour='white',
                    cmap='afmhot',
                    vmin=0,vmax=0.2,
                    fig=fig,
                    ax=ax4)
    
    for ax in [ax3,ax4]:

        ax.plot((x0,x1),(y0,y1),'w-',linewidth=2)
        ax.plot(x0,y0,'x',markeredgecolor='w')
        ax.plot(x1,y1,'o',markeredgecolor='w',markerfacecolor='k')
        
    #     # if x1 > x0: 
    #     #     ax.set_xlim([ x1 - 100, x0 + 100])
    #     # else: 
    #     #     ax.set_xlim([ x0+100, x1 - 100])

    #     # if y1 > y0: 
    #     #     ax.set_ylim([ y1 - 100, y0 + 100])
    #     # else: 
    #     #     ax.set_ylim([ y0+100, y1 - 100])

        ax.set_xlim([min(x0,x1)-100,max(x0,x1)+100])
        ax.set_ylim([max(y0,y1)+100,min(y0,y1)-100])

    plt.tight_layout()

    return pts


In [175]:
new_pts = height_map_line_profiler(dic_map,click_on=True,line_profile_width=3,points=ok_pts)

In [ ]:
ok_pts1 = np.array([[2009.35227273, 1251.99350649],
       [2106.65746753, 1130.36201299]])

In [ ]:
ok_pts2 = np.array([[1522.8262987 , 1333.08116883],
       [1644.45779221, 1333.08116883]])